In [1]:
import numpy as np
import pandas as pd
import os

os.chdir('C://Users/manni/Desktop')

In [2]:
data = pd.read_csv("Merged Rainfall Data.csv", parse_dates=[0])
data

,date,adm2_id,ADM2_PCODE,n_pixels,rfh,rfh_avg,r1h,r1h_avg,r3h,r3h_avg,rfq,r1q,r3q,version,ADM2_EN,SENDIST_EN,ADM1_EN
0,2021-01-01,22980,NG029014,28.0,4.9286,3.2274,11.7857,9.9821,180.4643,220.9571,120.6772,109.0259,82.4674,final,Okitipupa,Ondo South,Ondo
1,2021-01-11,22980,NG029014,28.0,2.2143,3.7071,9.1429,10.3190,134.7500,149.4417,82.8548,94.2114,90.7856,final,Okitipupa,Ondo South,Ondo
2,2021-01-21,22980,NG029014,28.0,1.4643,6.0607,8.6071,12.9952,65.7500,82.5226,58.4437,80.9174,81.8719,final,Okitipupa,Ondo South,Ondo
3,2021-02-01,22980,NG029014,28.0,5.2143,9.3012,8.8929,19.0690,36.0357,62.3286,71.4226,64.9930,63.6480,final,Okitipupa,Ondo South,Ondo
4,2021-02-11,22980,NG029014,28.0,12.7143,15.7690,19.3929,31.1310,41.1429,63.6548,85.2918,71.4616,69.4359,final,Okitipupa,Ondo South,Ondo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123763,2025-05-11,23001,NG030013,33.0,33.6364,55.8818,127.5455,155.6455,364.3636,344.8535,63.4613,83.0361,105.4981,final,Ife North,Osun East,Osun
123764,2025-05-21,23001,NG030013,33.0,37.6061,62.2465,94.5152,173.8182,382.2121,395.3495,63.3581,56.8579,96.7590,final,Ife North,Osun East,Osun
123765,2025-06-01,23001,NG030013,33.0,86.4545,91.7434,157.6970,209.8717,451.2727,463.5848,94.5331,76.2704,97.4002,prelim,Ife North,Osun East,Osun
123766,2025-06-11,23001,NG030013,33.0,68.5152,83.6848,192.5758,237.6747,462.6970,514.2283,82.8948,81.7910,90.1701,prelim,Ife North,Osun East,Osun


In [3]:
data.columns

Index(['date', 'adm2_id', 'ADM2_PCODE', 'n_pixels', 'rfh', 'rfh_avg', 'r1h',
       'r1h_avg', 'r3h', 'r3h_avg', 'rfq', 'r1q', 'r3q', 'version', 'ADM2_EN',
       'SENDIST_EN', 'ADM1_EN'],
      dtype='object')

In [4]:
df = data[data['ADM2_EN'] == "Kabba/Bunu"]
df = df[['date', 'rfh']]

df.set_index('date', inplace=True)

# --- Step 3: Resample to daily frequency and interpolate ---
# Create a new date range from the minimum to the maximum date in your data, daily frequency
full_date_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq='D')

# Reindex the DataFrame to this full daily range, which will introduce NaNs for missing days
df_daily = df.reindex(full_date_range)

# Apply linear interpolation to fill the NaN values
df_interpolated = df_daily.interpolate(method='linear')
df = df_interpolated.reset_index()
df.columns = ['date', 'rfh']
df

,date,rfh
0,2021-01-01,1.20690
1,2021-01-02,1.18621
2,2021-01-03,1.16552
3,2021-01-04,1.14483
4,2021-01-05,1.12414
...,...,...
1628,2025-06-17,68.51956
1629,2025-06-18,67.59082
1630,2025-06-19,66.66208
1631,2025-06-20,65.73334


In [5]:
import plotly.express as px
px.line(df, x = 'date', y = 'rfh', title='Rainfall Data', labels={'date': 'Date', 'rfh': 'Rainfall Height (mm)'})

In [6]:
df['day'] = df['date'].dt.day
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year
df['day_of_year'] = df['date'].dt.dayofyear
df['week_of_year'] = df.day_of_year//7 + 1

df

,date,rfh,day,month,year,day_of_year,week_of_year
0,2021-01-01,1.20690,1,1,2021,1,1
1,2021-01-02,1.18621,2,1,2021,2,1
2,2021-01-03,1.16552,3,1,2021,3,1
3,2021-01-04,1.14483,4,1,2021,4,1
4,2021-01-05,1.12414,5,1,2021,5,1
...,...,...,...,...,...,...,...
1628,2025-06-17,68.51956,17,6,2025,168,25
1629,2025-06-18,67.59082,18,6,2025,169,25
1630,2025-06-19,66.66208,19,6,2025,170,25
1631,2025-06-20,65.73334,20,6,2025,171,25


In [7]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score

X = df.drop(['date', 'day_of_year', 'rfh'], axis = 1)
y = df['rfh']

split_date = '2025-01-01'
# Try splitting 80% train, 20% test based on number of rows if a date split is problematic
split_idx = int(0.9*len(X))          #len(df[df['datetime'] <= split_date])

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# Training
model = RandomForestRegressor(n_estimators = 50, random_state = 42, n_jobs= -1, min_samples_leaf=8)
# model = GradientBoostingRegressor(random_state=42, n_estimators=100, learning_rate=0.1, max_depth=10)
model.fit(X_train, y_train)

#Accuracy
train_accuracy = model.score(X_train, y_train)
test_accuracy = model.score(X_test, y_test)

train_accuracy, test_accuracy

(0.9501815804475838, 0.7356153927216159)

In [8]:
predictions = model.predict(X_test)

import plotly.express as px
import plotly.graph_objects as go # Needed for go.Scatter to add traces


fig = px.line(
    x=df['date'][:len(y_train)],
    y=y_train,
    labels={'x': 'Date', 'y': 'Value'}, # Add labels for clarity
    title='Time Series: Training, Test, and Predictions'
)

# 2. Add the actual test data as a new trace
fig.add_trace(
    go.Scatter(
        x=df['date'][len(y_train):],
        y=y_test,
        mode='lines',
        name='Actual Test Data', # Label for the legend
        line=dict(color='orange') # Optional: customize line color
    )
)

# 3. Add the predictions as another new trace
#    The x-axis for predictions should align with the test data.
fig.add_trace(
    go.Scatter(
        x=df['date'][len(y_train):],
        y=predictions,
        mode='lines',
        name='Model Predictions', # Label for the legend
        line=dict(color='green', dash='dot') # Optional: customize line color and style
    )
)

# Optional: Update layout for better readability
fig.update_layout(
    hovermode="x unified", # Shows all y-values at a given x-point on hover
    xaxis_title="Date",
    yaxis_title="Rainfall Height (mm)",
)

# Show the figure
fig.show()


In [9]:
from datetime import date

lga = "Kabba/Bunu"
df = data[data['ADM2_EN'] == lga]
df = df[['date', 'rfh']]

df.set_index('date', inplace=True)

full_date_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq='D')

# Reindex the DataFrame to this full daily range, which will introduce NaNs for missing days
df_daily = df.reindex(full_date_range)

# Apply linear interpolation to fill the NaN values
df_interpolated = df_daily.interpolate(method='linear')
df = df_interpolated.reset_index()
df.columns = ['date', 'rfh']

df = df.reset_index().drop('index', axis = 1)

# Preprocessing
df['day'] = df['date'].dt.day
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year
df['day_of_year'] = df['date'].dt.dayofyear
df['week_of_year'] = df.day_of_year//7 + 1

X = df.drop(['date', 'day_of_year', 'rfh'], axis = 1)
y = df['rfh']


# Training
model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
model.fit(X, y)


# Forecasting
forecast_start_date = df['date'].max()
forecast_end_date = forecast_start_date + pd.Timedelta(days = 365)
forecast_dates = pd.date_range(start=forecast_start_date, end=forecast_end_date, freq='D')
#print(f"Forecasting from {forecast_dates.min()} to {forecast_dates.max()} ({len(forecast_dates)} days).")
# Forecasting
forecast_start_date = df['date'].max()
forecast_dates = pd.date_range(start=forecast_start_date, end=date(2025,12,31), freq='D')

df1 = pd.DataFrame(forecast_dates)
df1.columns = ['date']
df1['day'] = df1['date'].dt.day
df1['month'] = df1['date'].dt.month
df1['year'] = df1['date'].dt.year
df1['day_of_year'] = df1['date'].dt.dayofyear
df1['week_of_year'] = df1.day_of_year//7 + 1
df1 = df1.drop(['date', 'day_of_year'], axis = 1)
prediction = model.predict(df1)

actual = df[df['date'] >= "2024-01-01"].copy()
actual['type'] = 'Actual'

pred = pd.DataFrame(forecast_dates)
pred.columns = ['date']
pred['rfh'] = prediction
pred['type'] = 'Forecasted'

full_df = pd.concat([actual, pred], axis = 0)
full_df['lga'] = lga

full_df = full_df[['lga','date', 'rfh', 'type']]
full_df

#forecast_df = pd.concat([forecast_df, full_df], axis = 0)


,lga,date,rfh,type
1095,Kabba/Bunu,2024-01-01,1.287400,Actual
1096,Kabba/Bunu,2024-01-02,1.258660,Actual
1097,Kabba/Bunu,2024-01-03,1.229920,Actual
1098,Kabba/Bunu,2024-01-04,1.201180,Actual
1099,Kabba/Bunu,2024-01-05,1.172440,Actual
...,...,...,...,...
189,Kabba/Bunu,2025-12-27,1.481176,Forecasted
190,Kabba/Bunu,2025-12-28,1.471130,Forecasted
191,Kabba/Bunu,2025-12-29,1.473491,Forecasted
192,Kabba/Bunu,2025-12-30,1.237887,Forecasted


In [ ]:
bfig = px.line(full_df[full_df['type'] == 'Actual'],
    x = 'date',
    y = 'rfh',
              color = px.Constant('Actual Data'),
    labels = {'date': 'Date', 'rfh': 'Rainfall Height (mm)', 'color':'Legend'}, # Add labels for clarity
    title = 'Actual Data and Forecast'
)

# 2. Add the actual test data as a new trace
fig.add_trace(
    go.Scatter(
        x=full_df[full_df['type'] == 'Forecasted']['date'],
        y=full_df[full_df['type'] == 'Forecasted']['rfh'],
        mode='lines',
        name='Forecasted data', # Label for the legend
        line=dict(color='yellow') # Optional: customize line color
    )
)


# Show the figure
fig.show()

In [11]:
pd.read_csv("Merged Rainfall Data.csv", parse_dates=[0])

,date,adm2_id,ADM2_PCODE,n_pixels,rfh,rfh_avg,r1h,r1h_avg,r3h,r3h_avg,rfq,r1q,r3q,version,ADM2_EN,SENDIST_EN,ADM1_EN
0,2021-01-01,22980,NG029014,28.0,4.9286,3.2274,11.7857,9.9821,180.4643,220.9571,120.6772,109.0259,82.4674,final,Okitipupa,Ondo South,Ondo
1,2021-01-11,22980,NG029014,28.0,2.2143,3.7071,9.1429,10.3190,134.7500,149.4417,82.8548,94.2114,90.7856,final,Okitipupa,Ondo South,Ondo
2,2021-01-21,22980,NG029014,28.0,1.4643,6.0607,8.6071,12.9952,65.7500,82.5226,58.4437,80.9174,81.8719,final,Okitipupa,Ondo South,Ondo
3,2021-02-01,22980,NG029014,28.0,5.2143,9.3012,8.8929,19.0690,36.0357,62.3286,71.4226,64.9930,63.6480,final,Okitipupa,Ondo South,Ondo
4,2021-02-11,22980,NG029014,28.0,12.7143,15.7690,19.3929,31.1310,41.1429,63.6548,85.2918,71.4616,69.4359,final,Okitipupa,Ondo South,Ondo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123763,2025-05-11,23001,NG030013,33.0,33.6364,55.8818,127.5455,155.6455,364.3636,344.8535,63.4613,83.0361,105.4981,final,Ife North,Osun East,Osun
123764,2025-05-21,23001,NG030013,33.0,37.6061,62.2465,94.5152,173.8182,382.2121,395.3495,63.3581,56.8579,96.7590,final,Ife North,Osun East,Osun
123765,2025-06-01,23001,NG030013,33.0,86.4545,91.7434,157.6970,209.8717,451.2727,463.5848,94.5331,76.2704,97.4002,prelim,Ife North,Osun East,Osun
123766,2025-06-11,23001,NG030013,33.0,68.5152,83.6848,192.5758,237.6747,462.6970,514.2283,82.8948,81.7910,90.1701,prelim,Ife North,Osun East,Osun


In [12]:
from datetime import date

data = pd.read_csv("Merged Rainfall Data.csv", parse_dates=[0])
#data = data.drop_duplicates(keep='first')

forecast_df = pd.DataFrame()

for lga in data['ADM2_EN'].unique():
    try:
        df = data[data['ADM2_EN'] == lga]
        df = df[['date', 'rfh']]

        if df.duplicated(subset=['date']).all():
            df['date'] = pd.to_datetime(df['date'])
            df.drop_duplicates(['date'], inplace=True)

        df.set_index('date', inplace=True)

        # --- Step 3: Resample to daily frequency and interpolate ---
        # Create a new date range from the minimum to the maximum date in your data, daily frequency
        full_date_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq='D')
        full_date_range = full_date_range.drop_duplicates(keep='first')

        # Reindex the DataFrame to this full daily range, which will introduce NaNs for missing days
        df_daily = df.reindex(full_date_range)

        # Apply linear interpolation to fill the NaN values
        df_interpolated = df_daily.interpolate(method='linear')
        df = df_interpolated.reset_index()
        df.columns = ['date', 'rfh']

        df = data[data['ADM2_EN'] == lga]
        df = df[['date', 'rfh']]

        df.set_index('date', inplace=True)

        full_date_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq='D')
        full_date_range.drop_duplicates()

        # Reindex the DataFrame to this full daily range, which will introduce NaNs for missing days
        df_daily = df.reindex(full_date_range)

        # Apply linear interpolation to fill the NaN values
        df_interpolated = df_daily.interpolate(method='linear')
        df = df_interpolated.reset_index()
        df.columns = ['date', 'rfh']

        df = df.reset_index().drop('index', axis = 1)

        # Preprocessing
        df['day'] = df['date'].dt.day
        df['month'] = df['date'].dt.month
        df['year'] = df['date'].dt.year
        df['day_of_year'] = df['date'].dt.dayofyear
        df['week_of_year'] = df.day_of_year//7 + 1

        X = df.drop(['date', 'day_of_year', 'rfh'], axis = 1)
        y = df['rfh']


        # Training
        model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
        model.fit(X, y)


        # Forecasting
        forecast_start_date = df['date'].max()
        forecast_end_date = forecast_start_date + pd.Timedelta(days = 365)
        forecast_dates = pd.date_range(start=forecast_start_date, end=forecast_end_date, freq='D')
        #print(f"Forecasting from {forecast_dates.min()} to {forecast_dates.max()} ({len(forecast_dates)} days).")
        # Forecasting
        forecast_start_date = df['date'].max()
        forecast_dates = pd.date_range(start=forecast_start_date, end=date(2025,12,31), freq='D')

        df1 = pd.DataFrame(forecast_dates)
        df1.columns = ['date']
        df1['day'] = df1['date'].dt.day
        df1['month'] = df1['date'].dt.month
        df1['year'] = df1['date'].dt.year
        df1['day_of_year'] = df1['date'].dt.dayofyear
        df1['week_of_year'] = df1.day_of_year//7 + 1
        df1 = df1.drop(['date', 'day_of_year'], axis = 1)
        prediction = model.predict(df1)

        actual = df[df['date'] >= "2024-01-01"].copy()
        actual['type'] = 'Actual'

        pred = pd.DataFrame(forecast_dates)
        pred.columns = ['date']
        pred['rfh'] = prediction
        pred['type'] = 'Forecasted'

        full_df = pd.concat([actual, pred], axis = 0)
        full_df['lga'] = lga

        full_df = full_df[['lga','date', 'rfh', 'type']]

        forecast_df = pd.concat([forecast_df, full_df], axis = 0)
        print("Done with ", lga)
    except Exception as e:
        print("Stopped at", lga)
        print(e, "has",df.duplicated().sum(), "duplicate entries")
        break


Done with  Okitipupa
Done with  Mai'adua
Done with  Taura
Done with  Toungo
Done with  Tofa
Done with  Toto
Done with  Tsanyawa
Done with  Tudun Wada
Done with  Tureta
Done with  Udenu
Done with  Udu
Done with  Udung Uko
Done with  Ughelli North
Done with  Ughelli South
Done with  Ugwunagbo
Done with  Uhunmwonde
Done with  Ukwuani
Done with  Tarmua
Done with  Umuahia North
Done with  Tarka
Done with  Tangaza
Done with  Shendam
Done with  Shinkafi
Done with  Shira
Done with  Shiroro
Done with  Shomgom
Stopped at Kware
cannot reindex on an axis with duplicate labels has 191 duplicate entries


In [36]:
lgas = pd.read_csv('Adm_Data_Names.csv')
lgas_dict = dict(zip(lgas['ADM2_PCODE'], lgas['ADM2_EN']))
for lga, code in lgas_dict.items():
    print(lga, code)

NG001001 Aba North
NG001002 Aba South
NG008001 Abadam
NG015001 Abaji
NG003001 Abak
NG011001 Abakaliki
NG028001 Abeokuta North
NG028002 Abeokuta South
NG009001 Abi
NG017001 Aboh-Mbaise
NG033001 Abua/Odual
NG015002 Abuja Municipal
NG023001 Adavi
NG007001 Ado
NG028003 Ado-Odo/Ota
NG013001 Ado Ekiti
NG031001 Afijio
NG011002 Afikpo North
NG011003 Afikpo South
NG027001 Agaie
NG007002 Agatu
NG025001 Agege
NG004001 Aguata
NG027002 Agwara
NG017002 Ahiazu-Mbaise
NG033002 Ahoada East
NG033003 Ahoada West
NG030001 Aiyedade
NG030002 Aiyedire
NG013002 Aiyekire (Gbonyin)
NG023002 Ajaokuta
NG025002 Ajeromi-Ifelodun
NG020001 Ajingi
NG009002 Akamkpa
NG031002 Akinyele
NG016001 Akko
NG012001 Akoko-Edo
NG029001 Akoko North East
NG029002 Akoko North West
NG029003 Akoko South East
NG029004 Akoko South West
NG009003 Akpabuyo
NG033004 Akuku Toru
NG029005 Akure North
NG029006 Akure South
NG026001 Akwanga
NG020002 Albasu
NG022001 Aleiro
NG025003 Alimosho
NG005001 Alkaleri
NG025004 Amuwo-Odofin
NG004002 Anambra E

In [41]:
import pandas as pd
from datetime import date # Keep this import, though 'date(2025,12,31)' is likely the main use
from sklearn.ensemble import RandomForestRegressor

lgas = pd.read_csv('Adm_Data_Names.csv')
lgas_dict = dict(zip(lgas['ADM2_PCODE'], lgas['ADM2_EN']))
len(lags_dict)

# --- Configuration ---
DATA_FILE = "Merged Rainfall Data.csv"
TARGET_COLUMN = 'rfh'
LGA_COLUMN = 'ADM2_PCODE' # Assuming 'ADM2_EN' is the LGA identifier
VERSION_COLUMN = 'version' # Assuming you also have a 'version' column for filtering 'final'

# Initialize an empty DataFrame to store all forecasts
all_lga_forecast_df = pd.DataFrame()

# Load the full dataset once
data = pd.read_csv(DATA_FILE, parse_dates=[0]) # parse_dates=[0] means parse the first column as date

# Ensure the date column is datetime type for all operations
data['date'] = pd.to_datetime(data['date'])

# Iterate through each unique LGA
for lga in data[LGA_COLUMN].unique():                  #lga here is the lga code and the code is the lga name
    try:
        #print(f"\nProcessing LGA: {lga}")

        # --- Step 1: Filter Data for current LGA and 'final' version ---
        # Get data for the specific LGA
        df_lga = data[data[LGA_COLUMN] == lga].copy() # Use .copy() to avoid SettingWithCopyWarning

        # Filter for 'final' version data (if applicable and available in your CSV)
        # If your CSV doesn't have a 'version' column, you can remove this line
        if VERSION_COLUMN in df_lga.columns:
            df_lga = df_lga[df_lga[VERSION_COLUMN] == 'final'].copy()


        # --- Step 2: Prepare Dekadal Data for Daily Interpolation ---
        # Select only the date and target rainfall column
        df_lga_dekadal = df_lga[['date', TARGET_COLUMN]].copy()

        # Check for duplicates on the 'date' column within this LGA's dekadal data
        # If any duplicates exist for the same date, drop them (e.g., keep the first)
        if df_lga_dekadal.duplicated(subset=['date']).any():
            print(f"  Warning: Duplicate dates found for {lga}. Dropping duplicates, keeping first.")
            df_lga_dekadal.drop_duplicates(subset=['date'], inplace=True)

        # Set 'date' as the index
        df_lga_dekadal.set_index('date', inplace=True)

        # Sort the index to ensure chronological order (crucial for time series)
        df_lga_dekadal.sort_index(inplace=True)

        # Check if index is unique AFTER setting it and sorting
        if not df_lga_dekadal.index.is_unique:
             print(f"  CRITICAL ERROR: Index for {lga} is NOT UNIQUE after deduplication and setting. This should not happen.")
             # You might want to break here or log more detailed info if this occurs
             continue # Skip to next LGA if index is not unique

        # --- Step 3: Resample to daily frequency and interpolate ---
        # Create a new date range from the minimum to the maximum date in this LGA's data, daily frequency
        full_date_range = pd.date_range(start=df_lga_dekadal.index.min(),
                                        end=df_lga_dekadal.index.max(),
                                        freq='D')

        # Reindex the DataFrame to this full daily range, which will introduce NaNs for missing days
        # This is the line that caused the error, but should now work if df_lga_dekadal.index is unique
        df_daily_interpolated = df_lga_dekadal.reindex(full_date_range)

        # Apply linear interpolation to fill the NaN values
        df_daily_interpolated = df_daily_interpolated.interpolate(method='linear')

        # Reset index to make 'date' a column again for feature engineering
        df_processed = df_daily_interpolated.reset_index()
        df_processed.columns = ['date', TARGET_COLUMN] # Ensure column names are correct


        # --- Step 4: Feature Engineering for RandomForestRegressor ---
        df_processed['day'] = df_processed['date'].dt.day
        df_processed['month'] = df_processed['date'].dt.month
        df_processed['year'] = df_processed['date'].dt.year # Consider if 'year' is a good feature for generalization
        df_processed['day_of_year'] = df_processed['date'].dt.dayofyear
        df_processed['week_of_year'] = df_processed['day_of_year'] // 7 + 1


        # Define features (X) and target (y)
        # Assuming you want to predict 'rfh' based on date-derived features
        features_for_rf = ['day', 'month', 'year', 'week_of_year'] # day_of_year is often redundant with week_of_year, month, day
        X = df_processed[features_for_rf]
        y = df_processed[TARGET_COLUMN]

        # --- Step 5: Training the RandomForestRegressor ---
        model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
        model.fit(X, y)


        # --- Step 6: Forecasting ---
        # Define forecast period (e.g., from last historical date to end of next year)
        # Let's adjust forecast_start_date to be the day *after* the last historical day
        forecast_start_date = df_processed['date'].max() + pd.Timedelta(days=1)
        forecast_end_date = date(2025, 12, 31) # Forecasting up to end of 2025 as in your code

        forecast_dates = pd.date_range(start=forecast_start_date, end=forecast_end_date, freq='D')

        # Create DataFrame for forecast features
        df_forecast_features = pd.DataFrame(forecast_dates, columns=['date'])
        df_forecast_features['day'] = df_forecast_features['date'].dt.day
        df_forecast_features['month'] = df_forecast_features['date'].dt.month
        df_forecast_features['year'] = df_forecast_features['date'].dt.year
        df_forecast_features['day_of_year'] = df_forecast_features['date'].dt.dayofyear
        df_forecast_features['week_of_year'] = df_forecast_features['day_of_year'] // 7 + 1

        # Select only the features the model was trained on
        X_forecast = df_forecast_features[features_for_rf]

        # Make predictions
        predictions = model.predict(X_forecast)

        # --- Step 7: Combine Actual and Forecasted Data ---
        # Get actual data for plotting/comparison (from 2024-01-01 as in your code)
        actual_data_for_plot = df_processed[df_processed['date'] >= pd.to_datetime("2024-01-01")].copy()
        actual_data_for_plot['type'] = 'Actual'

        # Create DataFrame for forecasted results
        forecasted_data_df = pd.DataFrame({
            'date': forecast_dates,
            TARGET_COLUMN: predictions,
            'type': 'Forecasted'
        })

        # Concatenate actuals and forecasts for this LGA
        lga_full_df = pd.concat([actual_data_for_plot, forecasted_data_df], axis=0)
        lga_full_df[LGA_COLUMN] = lga # Add LGA identifier
        lga_full_df['ADM2_EN'] = lgas_dict[lga] # Add LGA identifier

        lga_full_df = lga_full_df[[LGA_COLUMN, 'ADM2_EN', 'date', TARGET_COLUMN, 'type']]


        # --- Step 8: Append to overall forecast DataFrame ---
        all_lga_forecast_df = pd.concat([all_lga_forecast_df, lga_full_df], axis=0)

        #print(f"  Successfully processed {lga}.")

    except Exception as e:
        print(f"  Stopped at LGA: {lga}")
        # Make sure df_lga_dekadal is defined before checking its index if error occurs early
        if 'df_lga_dekadal' in locals() and not df_lga_dekadal.index.is_unique:
             print(f"  Error likely due to non-unique index in {lga}. Index duplicates: {df_lga_dekadal.index[df_lga_dekadal.index.duplicated()].tolist()}")
        print(f"  Error message: {e}")
        # If you want to break the loop on first error, keep `break` here:
        # break
        # If you want to continue to the next LGA despite errors, remove `break`

In [42]:
all_lga_forecast_df

,ADM2_PCODE,ADM2_EN,date,rfh,type
1095,NG029014,Okitipupa,2024-01-01,1.821400,Actual
1096,NG029014,Okitipupa,2024-01-02,1.739260,Actual
1097,NG029014,Okitipupa,2024-01-03,1.657120,Actual
1098,NG029014,Okitipupa,2024-01-04,1.574980,Actual
1099,NG029014,Okitipupa,2024-01-05,1.492840,Actual
...,...,...,...,...,...
219,NG030013,Ife North,2025-12-27,1.004018,Forecasted
220,NG030013,Ife North,2025-12-28,1.000644,Forecasted
221,NG030013,Ife North,2025-12-29,1.000644,Forecasted
222,NG030013,Ife North,2025-12-30,1.000644,Forecasted


In [43]:
all_lga_forecast_df.to_csv('Rainfall Forecast.csv', index=False)

In [99]:
!dir

 Volume in drive C has no label.
 Volume Serial Number is 4828-5060

 Directory of C:\Users\manni\Desktop

07/10/2025  02:49 PM    <DIR>          .
07/10/2025  08:03 AM    <DIR>          ..
06/27/2025  05:57 PM    <DIR>          .ipynb_checkpoints
07/08/2025  02:05 PM            84,369 Adm_Data_Names.csv
06/26/2025  10:31 AM        26,111,715 Append1.xlsx
07/01/2025  08:16 AM        12,802,968 Approved Sid-Trac Profile complete.pdf
06/10/2025  06:51 AM    <DIR>          Arewa DS Project
06/22/2025  03:56 AM    <DIR>          Aud
05/14/2025  06:05 PM        94,759,766 Brand_Assets[1].zip
06/19/2025  11:14 AM             1,402 CapCut.lnk
05/14/2025  07:57 PM             2,403 Carnac.lnk
06/26/2025  10:29 AM        49,273,954 Different States.xlsx
05/13/2025  10:55 AM             1,091 DroidCamApp.lnk
05/31/2025  04:16 AM    <DIR>          EPL Project
06/09/2025  07:01 PM             2,369 GitHub Desktop.lnk
07/06/2025  01:20 PM             2,278 Google Chrome.lnk
07/09/2025  02:48 PM    